# IngredientMassEstimation V6 — ASL, escalas por clase, depth espacial y CLIP

Implementación nueva de TODOS los cambios: ASL sin corrección BCE, solo etapa 4 ajustable,
cantidades por escala train de cada clase, dos compuertas espaciales y consultas inicializadas con texto CLIP.
Se mantiene MLP compartido y masa auxiliar de V5. RGB sigue siendo el único input externo.

El filtro de masa está activo al evaluar/inferir (inicial τ=0,05), NO durante training.
Se calibra en validación con guardas de masa/calorías; puede elegirse τ=0 si recortar perjudica las métricas.
No hay garantía de eliminar todos los falsos ingredientes ni de superar V5.

Todos los controles RUN están apagados. Usar sesión Colab nueva; no actualizar dependencias mientras otra corrida entrena.


## 1. Montar Drive

Mantener el acceso directo a la carpeta compartida en Mi unidad, o adaptar DRIVE en punto 3.


In [1]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, hashlib
from IPython.display import display
from google.colab import drive
drive.mount("/content/drive")



Mounted at /content/drive


## 2. Clonar la rama V6 publicada

Publicar primero v6-asl-spatial-clip. No subir notebooks anteriores ejecutados. Repositorio privado: PRIVATE_REPO=True y secreto GITHUB_TOKEN.


In [2]:
URL = "https://github.com/marcogloazzo/IngredientMassEstimation.git"
BRANCH = "v6-asl-spatial-clip"
PRIVATE_REPO = False
REPO = Path("/content/nutrition-code-v6")

def git_private(command):
    environment = os.environ.copy()
    try:
        if PRIVATE_REPO:
            from google.colab import userdata
            import base64
            token = userdata.get("GITHUB_TOKEN")
            authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            environment.update({
                "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {authorization}",
            })
            del token, authorization
        result = subprocess.run(command, env=environment, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("Git falló: revisá URL, rama publicada y permisos del token. No se imprime la credencial.")
        return result.stdout.strip()
    finally:
        environment.clear()

if (REPO / ".git").is_dir():
    current = git_private(["git", "-C", str(REPO), "branch", "--show-current"])
    assert current == BRANCH, f"Esta carpeta tiene otra rama: {current}. Usá una carpeta nueva."
    git_private(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH])
else:
    git_private(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)])

PROJECT = next((p for p in (REPO, REPO / "IngredientMassEstimation")
                if (p / "config/query2label_100_v6.json").is_file()), None)
assert PROJECT is not None, "Falta publicar el código V6."
os.chdir(PROJECT)
print("Proyecto:", PROJECT)
print("Rama:", git_private(["git", "-C", str(REPO), "branch", "--show-current"]))

print("Commit:", git_private(["git", "-C", str(REPO), "rev-parse", "HEAD"]))




Proyecto: /content/nutrition-code-v6
Rama: v6-asl-spatial-clip
Commit: 1d670d7358132b475ed6e7a198c7803f7b8ad92a


## 3. Dependencias y rutas aisladas

Reutiliza datos/mapas V3 en lectura. El helper de gráficos se carga desde archivo para evitar imports de paquetes editables o versiones viejas en el kernel. No se muestran páginas HTML.


In [3]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-v6.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
SOURCE = PROJECT / "src"
assert (SOURCE / "ingredient_mass").is_dir(), "Falta código: revisar punto 2."
sys.path.insert(0, str(SOURCE))
import importlib.util
helper = SOURCE / "ingredient_mass/notebook_reports.py"
assert helper.is_file(), "Falta publicar notebook_reports.py en Git."
spec = importlib.util.spec_from_file_location("v6_native_reports", helper)
reports = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reports)
show_saved_report = reports.show_saved_report
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DRIVE = Path("/content/drive/MyDrive/IngredientMassEstimation")
SOURCE_V3 = DRIVE / "v3"
SOURCE_V4 = DRIVE / "v4"
SOURCE_V5 = DRIVE / "v5"
V6 = DRIVE / "v6"
EXPERIMENT_TAG = "run_01"
REPORT_TAG = "eval_01"
assert all("/" not in tag and "\\" not in tag and tag not in ("", ".", "..")
           for tag in (EXPERIMENT_TAG, REPORT_TAG))
CONFIG = PROJECT / "config/query2label_100_v6.json"
cfg = json.loads(CONFIG.read_text())
assert cfg["presence_loss"] == "asl" and cfg["encoder_training"] == "last_stage"
RUN = V6 / "runs" / f"query2label_100_v6_asl_{EXPERIMENT_TAG}"
VALIDATION = V6 / "reports" / f"{EXPERIMENT_TAG}_{REPORT_TAG}" / "validation"
TEST = VALIDATION.parent / "test"
CHECKPOINT = VALIDATION / "calibrated.pt"
BASELINES = {"v3": SOURCE_V3 / "reports/run_01/validation",
             "v4_split_only": SOURCE_V4 / "reports/split_only_run_01_eval_01/validation",
             "v5": SOURCE_V5 / "reports/run_01_eval_01/validation"}
LABELS_PARENT = Path("/content/v6_prepared")
PROCESSED = LABELS_PARENT / "processed_100_grouped_provisional"
DATA_PARENT = Path("/content/nutrition5k_v3")
DEPTH_CACHE = Path("/content/depth_v3_320")
DEPTH_ARCHIVE = SOURCE_V3 / "depth_v3_320.zip"
CLIP_QUERIES = V6 / "weights/clip_queries.npz"
CLIP_CACHE = V6 / "weights/clip"

def run_stream(arguments):
    command = [str(arg) for arg in arguments]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

print("Device:", DEVICE, "| V6: ASL + solo etapa 4 + escalas por clase + depth espacial + CLIP")
print("RUN nuevo:", RUN)
print("Filtro en evaluación/inferencia:", cfg["mass_filter"])
print("La masa auxiliar no reemplaza ni reescala los gramos.")


Device: cuda | V6: ASL + solo etapa 4 + escalas por clase + depth espacial + CLIP
RUN nuevo: /content/drive/MyDrive/IngredientMassEstimation/v6/runs/query2label_100_v6_asl_run_01
Filtro en evaluación/inferencia: {'enabled': True, 'threshold': 0.05, 'apply_during_training': False, 'calibration_max_mass_increase': 0.05, 'calibration_max_calorie_increase': 0.05}
La masa auxiliar no reemplaza ni reescala los gramos.


## 4. Restaurar las etiquetas originales de V3

Mismo split provisional y vocabulario; no volver a preparar etiquetas. Verificar los hashes del paquete.


In [4]:
from pathlib import PurePosixPath
import stat

def extract_zip_safe(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        checked = []
        for info in source.infolist():
            name = info.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            if relative.is_absolute() or not relative.parts or ":" in relative.parts[0]:
                raise ValueError(f"Nombre inseguro en ZIP: {name}")
            target = (destination / relative).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Ruta insegura en ZIP: {name}")
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f"No se aceptan enlaces en ZIP: {name}")
            checked.append((info, target, name.endswith("/")))
        for info, target, is_directory in checked:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(info) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)


labels_zip = SOURCE_V3 / "data/processed_100_grouped_provisional.zip"
assert labels_zip.is_file(), f"Falta el paquete existente de V3: {labels_zip}"
if not (PROCESSED / "records.jsonl").is_file():
    extract_zip_safe(labels_zip, LABELS_PARENT)
package = json.loads((LABELS_PARENT / "package_manifest.json").read_text())
for name, digest in package["file_sha256"].items():
    path = PROCESSED / name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Etiquetas modificadas: {name}"
print("Etiquetas V3 conservadas:", PROCESSED)




Etiquetas V3 conservadas: /content/v6_prepared/processed_100_grouped_provisional


## 5. Preparar consultas CLIP — NO es entrenamiento

Primera sesión: RUN_CLIP_PREPARATION=True. Descarga los pesos oficiales una vez y codifica únicamente
los nombres de los ingredientes. Guarda clip_queries.npz en v6/weights/.
CLIP permanece congelado durante esta preparación; las consultas y su proyección se ajustan después con el modelo.
Other no usa una descripción CLIP genérica: conserva su embedding aleatorio entrenable.
Sesiones siguientes: la caché se verifica y reutiliza. No hay CLIP ejecutándose durante inferencia.


In [5]:
RUN_CLIP_PREPARATION = False
if RUN_CLIP_PREPARATION:
    run_stream([sys.executable, "-u", "scripts/prepare_clip_queries.py", "--config", CONFIG,
                "--vocabulary", PROCESSED / "vocabulary.json", "--output", CLIP_QUERIES,
                "--clip-cache", CLIP_CACHE, "--device", "cpu"])
if CLIP_QUERIES.is_file():
    run_stream([sys.executable, "-u", "scripts/prepare_clip_queries.py", "--config", CONFIG,
                "--vocabulary", PROCESSED / "vocabulary.json", "--output", CLIP_QUERIES,
                "--clip-cache", CLIP_CACHE, "--check-cache"])
else:
    print("Falta inicialización CLIP: habilitar RUN_CLIP_PREPARATION antes de entrenar.")


Verified CLIP cache: 99 explicit ingredients; other remains trainable/random
{
  "schema_version": 1,
  "model": "ViT-B/32",
  "repo_revision": "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6",
  "template": "a photo of {name}.",
  "vocabulary_sha256": "61bcc0b71d4ec1deebdba2279fc0408bbb28912e5c55a6c525dec22d4116cc21",
  "explicit_classes": 99,
  "other_policy": "random_trainable",
  "embedding_dim": 512,
  "text_prompts": [
    "a photo of olive oil.",
    "a photo of salt.",
    "a photo of garlic.",
    "a photo of pepper.",
    "a photo of onions.",
    "a photo of vinegar.",
    "a photo of lemon juice.",
    "a photo of cherry tomatoes.",
    "a photo of arugula.",
    "a photo of parsley.",
    "a photo of carrot.",
    "a photo of shallots.",
    "a photo of broccoli.",
    "a photo of spinach (raw).",
    "a photo of cucumbers.",
    "a photo of mustard.",
    "a photo of mixed greens.",
    "a photo of chicken.",
    "a photo of thyme.",
    "a photo of wheat berry.",
    "a photo 

## 6. Restaurar RGB y reutilizar el cache V3 320×320

No regenerar depth. RUN_DATA_CHECK=True verifica RGB, grupos, hashes de depth y consultas CLIP sin ejecutar el modelo.


In [6]:
def dataset_roots():
    if not DATA_PARENT.is_dir():
        return []
    return sorted({folder.parent for folder in DATA_PARENT.rglob("metadata")
                   if folder.is_dir()
                   and (folder / "dish_metadata_cafe1.csv").is_file()
                   and (folder / "ingredients_metadata.csv").is_file()
                   and (folder.parent / "dish_ids").is_dir()
                   and (folder.parent / "imagery/realsense_overhead").is_dir()})

roots = dataset_roots()
if not roots:
    source = DRIVE / "nutrition5k_rgb.zip"
    assert source.is_file(), f"Falta: {source}"
    local = Path("/content/nutrition5k_rgb_v6.zip")
    shutil.copy2(source, local)
    extract_zip_safe(local, DATA_PARENT)
    roots = dataset_roots()
assert len(roots) == 1, f"Raíz RGB no única: {roots}"
DATA = roots[0]
if not (DEPTH_CACHE / "manifest.json").is_file():
    assert DEPTH_ARCHIVE.is_file(), "Falta el ZIP de mapas V3 320x320 en Drive."
    local = Path("/content/depth_v3_320.zip")
    shutil.copy2(DEPTH_ARCHIVE, local)
    extract_zip_safe(local, DEPTH_CACHE.parent)
assert (DEPTH_CACHE / "manifest.json").is_file(), "Revisá la estructura del ZIP de depth V3."
RUN_DATA_CHECK = True
if RUN_DATA_CHECK:
    run_stream([sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG,
                "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
                "--clip-queries", CLIP_QUERIES, "--device", DEVICE, "--check-data"])
print("RGB:", DATA, "| cache compatible:", DEPTH_CACHE)


{
  "config": {
    "model_type": "convnext_small_query2label_spatial_clip_asl_v6",
    "seed": 42,
    "top_k": 100,
    "validation_fraction": 0.15,
    "split_policy": "grouped_provisional",
    "accept_provisional_split": true,
    "encoder": "convnext_small",
    "encoder_training": "last_stage",
    "image_size": 320,
    "attention_dim": 256,
    "attention_heads": 4,
    "attention_layers": 2,
    "attention_feedforward": 1024,
    "attention_dropout": 0.1,
    "shared_dims": [
      384,
      192
    ],
    "mass_hidden_dim": 128,
    "dropout": 0.3,
    "presence_loss": "asl",
    "presence_gate": "sigmoid",
    "conditional_huber_beta": 0.1,
    "split_task_mlps": false,
    "auxiliary_total_mass": true,
    "augmentation": {
      "rotation_degrees": 15,
      "brightness": 0.2,
      "contrast": 0.2,
      "saturation": 0.15,
      "hue": 0.03
    },
    "depth": {
      "backend": "depth_anything_v2",
      "encoder": "vits",
      "frozen": true,
      "input_size": 518

## 7. Entrenamiento V6 — desactivado

RUN_TRAINING=True solo cuando decidas entrenar. Primera corrida MODE="new", máximo 100 épocas y paciencia 8.
Durante las primeras 3 épocas el encoder está congelado; luego solo transición E3→E4, etapa 4 y norm final.
Stem/E1/E2/E3 quedan congelados/eval. No hay LoRA en esta variante.

ASL usa sigmoid sin restar log(pos_weight). Las escalas por clase se calculan SOLO con train y se guardan.
Se entrena con p×c sin corte para preservar gradientes; las métricas de validación usan el filtro.
History.csv informa también resultados SIN filtro y medios de las compuertas espaciales.
Resume solamente desde last.pt de esta V6, con la misma caché CLIP y configuración.


In [7]:
RUN_TRAINING = True
MODE = "resume"
ALLOW_CPU_TRAINING = False
TOTAL_EPOCHS = 100
if RUN_TRAINING:
    assert DEVICE == "cuda" or ALLOW_CPU_TRAINING, "Elegí GPU en Colab; CPU requiere ALLOW_CPU_TRAINING=True."
    assert CLIP_QUERIES.is_file(), "Primero preparar/verificar consultas CLIP en punto 5."
    assert MODE in ("new", "resume")
    args = [sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG,
            "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
            "--output-dir", RUN, "--torch-cache", DRIVE / "torch_cache",
            "--clip-queries", CLIP_QUERIES, "--device", DEVICE, "--workers", "2", "--epochs", TOTAL_EPOCHS]
    if MODE == "resume":
        assert (RUN / "last.pt").is_file(), "Falta last.pt de esta V6."
        args += ["--resume", RUN / "last.pt"]
    else:
        assert not RUN.exists() or not any(RUN.iterdir()), "RUN tiene resultados: usar resume de V6 o un EXPERIMENT_TAG nuevo."
    run_stream(args)
else:
    print("Entrenamiento V6 desactivado.")


Device: cuda; presence loss: asl; classes: 100
Model: convnext_small_query2label_spatial_clip_asl_v6; encoder adjustment: last_stage
Total-mass loss scale (training mean): 216.890021 g
Effective batch: 2 x 16; selection: ingredient_mass_l1_per_plate
Epoch 54: encoder last_stage trainable

Train 54: 100%|██████████| 1178/1178 [01:18<00:00, 15.02batch/s, loss=0.7569, samples=2355/2355]

Validation 54: 100%|██████████| 200/200 [00:06<00:00, 32.70batch/s, loss=1.4739, samples=400/400]
F1=0.562; ingredient L1/plate=185.2 g; mass MAE=63.4 g; calories MAE=96.6 kcal; epoch=85 s
Epoch 55: encoder last_stage trainable

Train 55: 100%|██████████| 1178/1178 [01:12<00:00, 16.31batch/s, loss=0.7552, samples=2355/2355]

Train 56: 100%|██████████| 1178/1178 [01:12<00:00, 16.19batch/s, loss=0.7474, samples=2355/2355]

Validation 56: 100%|██████████| 200/200 [00:06<00:00, 33.24batch/s, loss=1.4625, samples=400/400]
F1=0.561; ingredient L1/plate=182.6 g; mass MAE=63.1 g; calories MAE=90.6 kcal; epoch=79 

## 8. Validación y calibración; test apagado

RUN_CALIBRATION=True evalúa best.pt y calibra DOS umbrales distintos:
presencia por F1 y filtro de masa por L1, con guardas de MAE total/calorías respecto a no filtrar.
Se permite hasta +5% relativo de esas métricas (configurable); si conviene no recortar, puede seleccionarse τ=0.
La calibración de masa SÍ cambia gramos; ambas decisiones usan solo validación.
Usar REPORT_TAG nuevo al repetir. Dejar RUN_TEST=False durante selección de arquitectura/hiperparámetros.


In [ ]:
RUN_CALIBRATION = False
RUN_TEST = False
shared = ["--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
          "--device", DEVICE, "--batch-size", "2", "--workers", "2"]
if RUN_CALIBRATION:
    assert (RUN / "best.pt").is_file(), "Falta best.pt de esta V6."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", RUN / "best.pt",
                "--split", "validation", "--calibrate", "--output-dir", VALIDATION, *shared])
if RUN_TEST:
    assert CHECKPOINT.is_file(), "Primero calibrar con validación."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", CHECKPOINT,
                "--split", "test", "--output-dir", TEST, *shared])
for split, folder in (("VALIDACIÓN V6", VALIDATION), ("TEST V6 · solo referencia", TEST)):
    figures = show_saved_report(folder / "metrics.json", "metrics", split)
if not RUN_CALIBRATION and not RUN_TEST:
    print("Evaluación desactivada: solo se muestran reportes ya guardados.")



## 9. Diagnóstico y comparación V3/V4/V5 — solo archivos guardados

Tablas/gráficos nativos. Separar el efecto de p y del recorte, mirar métricas con y sin filtro. Las referencias que no estén disponibles se omiten; no se genera ni entrena un baseline automáticamente.


In [ ]:
RUN_V6_DIAGNOSTIC = False
RUN_COMPARISON = False
V6_DIAGNOSTIC = V6 / "diagnostics" / f"{EXPERIMENT_TAG}_{REPORT_TAG}"
if RUN_V6_DIAGNOSTIC:
    run_stream([sys.executable, "-u", "scripts/diagnose_mass_gate.py",
                "--predictions", VALIDATION / "predictions.npz",
                "--details", VALIDATION / "evaluation_details.json",
                "--vocabulary", PROCESSED / "vocabulary.json", "--output-dir", V6_DIAGNOSTIC])
figures = show_saved_report(V6_DIAGNOSTIC / "diagnostic.json", "diagnostic", "V6 · cantidad y filtro")
for label, previous in BASELINES.items():
    comparison_dir = V6 / "comparisons" / f"{label}_vs_v6_{EXPERIMENT_TAG}_{REPORT_TAG}"
    if RUN_COMPARISON:
        if not (previous / "metrics.json").is_file():
            print("Referencia todavía no disponible:", previous)
            continue
        run_stream([sys.executable, "-u", "scripts/compare_image_experiments.py",
                    "--left", previous, "--right", VALIDATION, "--output-dir", comparison_dir])
    figures = show_saved_report(comparison_dir / "comparison.json", "comparison", f"{label} vs V6")


## Git y Drive

Git: rama sugerida v6-asl-spatial-clip; código compartido compatible, robust_model.py, clip_queries.py,
mass_filtering.py, prepare_clip_queries.py, config V6, requirements-v6.txt, pruebas, documentos y este notebook limpio.
No seleccionar notebooks V3/V4/V5 ejecutados ni archivos de datos/pesos/resultados.

Drive: abrir/guardar este notebook. Reutilizar nutrition5k_rgb.zip, etiquetas agrupadas y depth_v3_320.zip.
La celda 5 creará los pesos/cache de CLIP en v6/weights/; no se suben a Git.
Resultados nuevos en v6/. No se modifican resultados V2–V5.
No hubo cambio de rama, commit, push, descargas de pesos ni training automático en la PC.


## 10. Inferencia RGB opcional y compuertas espaciales — apagada

Por defecto usar una imagen de validación. Las referencias reales se usan SOLO para mostrar, nunca como input.
El checkpoint calibrado ya contiene consultas CLIP y escalas por clase; solo Depth Anything se ejecuta desde RGB.
Atención no equivale a segmentación; las compuertas muestran cuánto depth se agrega en cada ubicación.
Usar INFERENCE_TAG nuevo para no sobrescribir predicciones.


In [ ]:
RUN_INFERENCE = False
PHOTO_SOURCE = "validation"  # validation / personal / test
PLATE_INDEX = 0
INFERENCE_TAG = "photo_01"
if RUN_INFERENCE:
    assert CHECKPOINT.is_file(), "Primero entrenar y calibrar V6."
    assert "/" not in INFERENCE_TAG and "\\" not in INFERENCE_TAG and INFERENCE_TAG not in ("", ".", "..")
    reference = None
    if PHOTO_SOURCE == "personal":
        PHOTO = DRIVE / "photos/IMG_9993.png"
    else:
        assert PHOTO_SOURCE in ("validation", "test")
        choices = [json.loads(line) for line in (PROCESSED / "records.jsonl").read_text().splitlines()
                   if json.loads(line)["split"] == PHOTO_SOURCE]
        assert 0 <= PLATE_INDEX < len(choices)
        reference = choices[PLATE_INDEX]
        PHOTO = DATA / reference["image"]
    assert PHOTO.is_file(), f"Falta RGB: {PHOTO}"
    DEPTH_REPO = Path("/content/depth-anything-v6")
    revision = cfg["depth"]["repo_revision"]
    if not (DEPTH_REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "https://github.com/DepthAnything/Depth-Anything-V2.git", DEPTH_REPO], check=True)
        subprocess.run(["git", "-C", DEPTH_REPO, "checkout", revision], check=True)
    assert subprocess.check_output(["git", "-C", DEPTH_REPO, "rev-parse", "HEAD"], text=True).strip() == revision
    DEPTH_WEIGHTS = SOURCE_V3 / "weights/depth_anything_v2_vits.pth"
    assert DEPTH_WEIGHTS.is_file(), f"Falta el peso depth ya usado en V3: {DEPTH_WEIGHTS}"
    inference_dir = V6 / "predictions" / f"{EXPERIMENT_TAG}_{REPORT_TAG}_{INFERENCE_TAG}"
    assert not inference_dir.exists() or not any(inference_dir.iterdir()), "Usar INFERENCE_TAG nuevo."
    run_stream([sys.executable, "-u", "scripts/infer_image_model.py", "--checkpoint", CHECKPOINT,
                "--image", PHOTO, "--output-dir", inference_dir, "--device", DEVICE,
                "--depth-anything-repo", DEPTH_REPO, "--depth-checkpoint", DEPTH_WEIGHTS,
                "--save-attention", "--top-k", "6"])
    saved = inference_dir / PHOTO.stem
    prediction = json.loads((saved / "prediction.json").read_text())
    rows = [{"metric": "Masa final filtrada (g)", "predicted": prediction["predicted_total_mass_grams"],
             "truth": reference["total_mass"] if reference else None},
            {"metric": "Masa sin filtro (g)", "predicted": prediction["unfiltered_total_mass_grams"], "truth": None},
            {"metric": "Masa auxiliar (g)", "predicted": prediction["auxiliary_total_mass_grams"], "truth": None}]
    for key, unit in (("calories","kcal"), ("fat","g"), ("carbohydrates","g"), ("protein","g")):
        rows.append({"metric": f"{key} ({unit})", "predicted": prediction["nutrition"][key],
                     "truth": reference["nutrients"][key] if reference else None})
    figure = reports._table("Referencia vs V6", rows, [("metric","Métrica"),("truth","Real"),("predicted","Estimado")])
    figure = reports._table("Ingredientes estimados · top 12", prediction["ingredients"][:12],
        [("name","Ingrediente"),("probability","p"),("conditional_grams","c (g)"),
         ("grams","Gramos filtrados"),("mass_retained","Pasa filtro")], width=14, first_width=.35)
    from PIL import Image
    import matplotlib.pyplot as plt
    images = [(saved / "intermediates/input_crop.png", "RGB del modelo"),
              (saved / "intermediates/estimated_relative_depth.png", "Depth relativo"),
              (saved / "intermediates/depth_spatial_gate_stage3.png", "Gate espacial E3"),
              (saved / "intermediates/depth_spatial_gate_stage4.png", "Gate espacial E4")]
    images += [(path, path.stem) for path in sorted((saved / "intermediates").glob("*_attention.png"))[:2]]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, (path, title) in zip(axes.flat, images):
        if path.is_file():
            with Image.open(path) as img:
                ax.imshow(img.copy())
        ax.set_title(title); ax.axis("off")
    for ax in list(axes.flat)[len(images):]:
        ax.axis("off")
    fig.tight_layout(); plt.show(); plt.close(fig)
else:
    print("Inferencia V6 desactivada.")
